# Real-World Graph Properties

## Learning Objectives

1. **Describe** small-world, scale-free, and community properties of real networks
2. **Compute** diameter, clustering coefficient, and degree distribution
3. **Generate** Watts-Strogatz and Barabási-Albert graphs
4. **Explain** why power-law degree distributions arise from preferential attachment

## Why Study Graph Properties?

Real-world networks — the Web, social networks, protein interactions, citation graphs — share surprising structural properties that distinguish them from random graphs.

Understanding these properties is prerequisite for algorithms that exploit graph structure (PageRank, community detection, GNNs).

**Three key properties:**
1. **Small world:** short average path length (6 degrees of separation)
2. **Scale-free:** power-law degree distribution P(k) ∝ k^{−γ}
3. **Community structure:** dense clusters with sparse inter-cluster edges

In [ ]:
import random, math
from collections import defaultdict, deque

def make_random_graph(n, p, seed=42):
    """Erdős-Rényi G(n,p): each edge exists independently with probability p."""
    rng = random.Random(seed)
    adj = defaultdict(set)
    for i in range(n):
        for j in range(i+1, n):
            if rng.random() < p:
                adj[i].add(j); adj[j].add(i)
    return adj

def make_watts_strogatz(n, k, beta, seed=42):
    """Watts-Strogatz small-world graph: ring lattice + rewiring."""
    rng = random.Random(seed)
    adj = defaultdict(set)
    # Ring lattice
    for i in range(n):
        for delta in range(1, k//2+1):
            j = (i+delta) % n
            adj[i].add(j); adj[j].add(i)
    # Rewire
    for i in range(n):
        for delta in range(1, k//2+1):
            j = (i+delta) % n
            if rng.random() < beta:
                adj[i].discard(j); adj[j].discard(i)
                new_j = rng.randint(0,n-1)
                while new_j == i or new_j in adj[i]:
                    new_j = rng.randint(0,n-1)
                adj[i].add(new_j); adj[new_j].add(i)
    return adj

def make_barabasi_albert(n, m, seed=42):
    """Barabási-Albert scale-free graph via preferential attachment."""
    rng = random.Random(seed)
    adj = defaultdict(set)
    # Start with a small complete graph
    for i in range(m+1):
        for j in range(i+1, m+1):
            adj[i].add(j); adj[j].add(i)
    degree_list = []
    for i in range(m+1):
        degree_list.extend([i]*(m))
    for i in range(m+1, n):
        targets = set()
        while len(targets) < m:
            targets.add(rng.choice(degree_list))
        for t in targets:
            adj[i].add(t); adj[t].add(i)
        degree_list.extend([i]*m)
        degree_list.extend(list(targets))
    return adj

n = 200
g_random = make_random_graph(n, 0.05)
g_ws     = make_watts_strogatz(n, 6, 0.1)
g_ba     = make_barabasi_albert(n, 3)
print("Graphs created:", n, "nodes each")
print(f"  Random edges: {sum(len(v) for v in g_random.values())//2}")
print(f"  WS edges:     {sum(len(v) for v in g_ws.values())//2}")
print(f"  BA edges:     {sum(len(v) for v in g_ba.values())//2}")

In [ ]:
def avg_clustering_coefficient(adj):
    """Local clustering coefficient averaged over all nodes."""
    total = 0.0; count = 0
    for u, neighbors in adj.items():
        k = len(neighbors)
        if k < 2: continue
        triangles = sum(1 for v in neighbors for w in neighbors if v < w and w in adj[v])
        total += 2*triangles / (k*(k-1))
        count += 1
    return total/count if count else 0.0

def approx_avg_path_length(adj, n_samples=100, seed=0):
    """Approximate average shortest path length via BFS from random sources."""
    nodes = list(adj.keys())
    rng = random.Random(seed)
    total = 0; pairs = 0
    for src in rng.sample(nodes, min(n_samples, len(nodes))):
        dist = {src: 0}; q = deque([src])
        while q:
            u = q.popleft()
            for v in adj[u]:
                if v not in dist:
                    dist[v] = dist[u]+1; q.append(v)
        for d in dist.values():
            total += d; pairs += 1
    return total/pairs if pairs else 0

def degree_distribution(adj):
    from collections import Counter
    return Counter(len(v) for v in adj.values())

print(f"{'Graph':12s} {'Avg path':10s} {'Clust coeff':12s} {'Max degree':10s}")
for name, g in [('Random',g_random),('WS',g_ws),('BA',g_ba)]:
    apl = approx_avg_path_length(g)
    cc  = avg_clustering_coefficient(g)
    max_deg = max(len(v) for v in g.values())
    print(f"{name:12s} {apl:10.3f} {cc:12.4f} {max_deg:10d}")

## Small-World Property

**Observation:** Real networks have short average path lengths (like random graphs) but high clustering coefficients (like regular lattices).

The Watts-Strogatz model captures this: start with a regular ring lattice (high clustering, long paths), then rewire a small fraction β of edges at random. Even β = 0.01 dramatically shortens paths while preserving most clusters.

**Six degrees of separation:** Milgram's 1967 experiment showed that any two Americans were connected by ~6 social links. Modern analysis of Facebook shows avg path ≈ 4.6.

## Scale-Free Networks and Power Laws

**Power-law degree distribution:** $P(k) \propto k^{-\gamma}$ with $\gamma \in [2, 3]$ for most real networks.

- **Consequence:** a few "hub" nodes have very high degree; most nodes have low degree
- **Heavy tail:** unlike random graphs where max degree ≈ O(log n), scale-free networks have hubs with degree O(n^{1/(γ−1)})

**Preferential attachment (Barabási-Albert):** new nodes attach to existing nodes with probability proportional to their current degree ("rich get richer"). This generates power-law distributions.

In [ ]:
import math
from collections import Counter

def fit_power_law_exponent(degree_dist, k_min=3):
    """MLE estimator for power law exponent (Clauset et al.)."""
    degrees = [k for k,c in degree_dist.items() for _ in range(c) if k >= k_min]
    if not degrees: return None
    n = len(degrees)
    gamma = 1 + n / sum(math.log(k/k_min) for k in degrees)
    return gamma

for name, g in [('Random',g_random),('WS',g_ws),('BA',g_ba)]:
    dd = degree_distribution(g)
    gamma = fit_power_law_exponent(dd)
    top5 = sorted(dd.items(), reverse=True)[:5]
    print(f"{name}: gamma={gamma:.2f if gamma else 'N/A'}, top degrees: {top5}")

## Summary

| Property | Random G(n,p) | Watts-Strogatz | Barabási-Albert |
|----------|--------------|----------------|-----------------|
| Avg path length | Low (log n) | Low | Low |
| Clustering coefficient | Low (~p) | **High** | Low-medium |
| Degree distribution | Poisson | Near-Poisson | **Power law** |
| Real-world fit | Poor | Partially | Partially |

Real networks exhibit **all three** properties simultaneously.
The **configuration model** (specify degree sequence, connect randomly) is a null model that fixes degree distribution while removing other structure.